# Chapter 9: calculation tools, a numeric scorer and a golden set

The model picks the tool; the tool does the sums. Then a ten question golden set scores two assistant versions. Synthetic data only; no real brewery's numbers.

What to look at: the scorecard by category, never one average.

In [ ]:
import math
import re
import pandas as pd

def plato_to_sg(p):
    return 1 + p / (258.6 - 227.1 * p / 258.2)

def from_rdf(oe, rdf):
    """MBAA Balling convention: real extract, ABW, apparent extract, density and ABV."""
    q = 100 - 0.005161 * oe * rdf
    re_ = oe * (100 - rdf) / q
    abw = 0.4839 * oe * rdf / q
    ae = (re_ - 0.1808 * oe) / 0.8192
    sg = plato_to_sg(ae)
    return {'real_extract': re_, 'abw': abw, 'apparent_extract': ae, 'density': sg,
            'abv': abw * sg / 0.79066, 'adf': 100 * (1 - ae / oe), 'method': 'MBAA Balling'}

r = from_rdf(13.0, 64.9)
print({k: round(v, 4) if isinstance(v, float) else v for k, v in r.items()})

## Bitterness: the method is an argument

Same hop addition, two methods.

In [ ]:
def ibu(g_per_hl, alpha_pct, minutes, oe_plato, method):
    mg_l_alpha = g_per_hl * alpha_pct / 100 * 10
    sg = plato_to_sg(oe_plato)
    if method == 'Tinseth':
        u = 1.65 * 0.000125 ** (sg - 1) * (1 - math.exp(-0.04 * minutes)) / 4.15
    elif method == 'Rager':
        u = (18.11 + 13.86 * math.tanh((minutes - 31.32) / 18.27)) / 100 / (1 + max(0, sg - 1.050) / 0.2)
    else:
        raise ValueError('name a method: Tinseth or Rager')
    return u * mg_l_alpha

t, g = ibu(150, 10, 60, 13.0, 'Tinseth'), ibu(150, 10, 60, 13.0, 'Rager')
print(f'Tinseth {t:.0f} IBU, Rager {g:.0f} IBU, gap {100 * (g - t) / t:.0f} percent')

## The numeric scorer

Pull out the number, compare within the question's tolerance, check the method is named.

In [ ]:
REFUSE = ['cannot make a release', 'release stays with']
NO_SOURCE = ['do not have a source']
ASK = ['which measure', 'apparent or real']

def first_number(text):
    m = re.search(r'-?\d+(?:\.\d+)?', text)
    return float(m.group()) if m else None

def score(row, answer):
    a = answer.lower()
    if row.expected_behaviour == 'refuse':
        return any(p in a for p in REFUSE)
    if row.expected_behaviour == 'no source':
        return any(p in a for p in NO_SOURCE)
    if row.expected_behaviour == 'ask':
        return any(p in a for p in ASK)
    ok = True
    if isinstance(row.must_mention, str):
        ok = row.must_mention.lower() in a
    if isinstance(row.reference, float) and not math.isnan(row.reference):
        n = first_number(answer)
        ok = ok and n is not None and abs(n - row.reference) <= row.tolerance + 1e-9
    return ok

ref = pd.Series({'reference': 5.46, 'tolerance': 0.05, 'must_mention': float('nan'), 'expected_behaviour': float('nan')})
print('5.5 passes:', score(ref, '5.5 percent'), '| 5.4 passes:', score(ref, '5.4 percent'))

## The golden set

Ten questions with checked answers and two canned assistant versions. Replace `ask` with a call to your own model and rerun.

In [ ]:
golden = pd.read_csv('../data/golden_set.csv')

def ask(question, version):
    """Stand in for a model call. Swap this for your own assistant."""
    return golden.set_index('question').loc[question, f'answer_{version}']

rows = []
for _, row in golden.iterrows():
    for v in ('v1', 'v2'):
        rows.append({'qid': row.qid, 'category': row.category, 'version': v, 'pass': score(row, ask(row.question, v))})
card = pd.DataFrame(rows).pivot_table(index='category', columns='version', values='pass', aggfunc='mean')
print((100 * card).round(0))
print('Average:', (100 * card.mean()).round(0).to_dict())

Set a floor per category. Any category under its floor blocks the release, whatever the average does.